<a href="https://colab.research.google.com/github/SofiiaBobr/goit_machine_learning/blob/main/hw3_bank_marketing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HW 03 — UCI Bank Marketing | Relational Databases for DS

**Джерело:** [UCI Machine Learning Repository, Bank Marketing (ID 222)](https://archive.ics.uci.edu/dataset/222/bank+marketing). Використано `bank-full.csv` (повна версія старого набору, 45 211 записів, 17 колонок). Завантаження з офіційного ZIP, фіксований вибір файлу, без випадкового семплінгу.

**Мета:** відтворюваний ETL `official ZIP → CSV → PostgreSQL staging (TEXT) → typed clean`, аудит, EDA та оцінка ML-ризиків. Усі висновки нижче обчислюються за фактичними результатами запитів. Запускайте **Runtime → Restart session and run all**.

## 1. Setup і офіційне завантаження

In [1]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas pyarrow ucimlrepo

import io, zipfile, urllib.request, pathlib, shutil
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

BASE = pathlib.Path('/content') if pathlib.Path('/content').exists() else pathlib.Path.cwd()
DATA_DIR = BASE / 'data'; DATA_DIR.mkdir(parents=True, exist_ok=True)
pg = pgserver.get_server('/tmp/hw3_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)
with engine.connect() as conn:
    print('PostgreSQL:', conn.execute(text('SELECT version()')).scalar_one())

OFFICIAL_URL = 'https://archive.ics.uci.edu/static/public/222/bank+marketing.zip'
with urllib.request.urlopen(OFFICIAL_URL, timeout=90) as response:
    payload = response.read()
with zipfile.ZipFile(io.BytesIO(payload)) as outer:
    names = outer.namelist()
    if 'bank.zip' in names:
        with zipfile.ZipFile(io.BytesIO(outer.read('bank.zip'))) as inner:
            csv_bytes = inner.read('bank-full.csv')
    elif 'bank-full.csv' in names:
        csv_bytes = outer.read('bank-full.csv')
    else:
        raise FileNotFoundError(f'Expected bank-full.csv/bank.zip; found: {names}')
CSV_PATH = DATA_DIR / 'bank-full.csv'
CSV_PATH.write_bytes(csv_bytes)
df_source = pd.read_csv(CSV_PATH, sep=';')
assert len(df_source) == 45211 and len(df_source.columns) == 17, df_source.shape
print('Official source:', OFFICIAL_URL)
print('CSV:', CSV_PATH, 'shape:', df_source.shape)
display(df_source.head())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 52.2 MB/s eta 0:00:00
PostgreSQL: PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit
Official source: https://archive.ics.uci.edu/static/public/222/bank+marketing.zip
CSV: /content/data/bank-full.csv shape: (45211, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


### Схема й типізація

| Поля | Семантика та сирий формат | Clean-тип і правило |
|---|---|---|
| age | вік, десятковий запис | SMALLINT; `18..100`, інакше відкинути |
| job, marital, education, contact, month, poutcome | категорії, TEXT | TEXT, `TRIM`; `unknown` → NULL для job/education/contact/poutcome; `unknown` у marital також NULL |
| default, housing, loan | yes/no | BOOLEAN: yes → TRUE, no → FALSE, інше → NULL |
| balance | баланс рахунку, ціле число | NUMERIC(14,2), від'ємні значення допустимі (борг) |
| day | день місяця | SMALLINT, 1..31 |
| duration | тривалість останнього контакту, секунди | INTEGER, ≥0; **leakage для прогнозу до дзвінка** |
| campaign, previous | кількість контактів | INTEGER, ≥0; campaign ≥1 |
| pdays | днів від попереднього контакту | INTEGER; -1 → NULL (не було контакту), решта ≥0 |
| y | підписка на депозит, yes/no | BOOLEAN NOT NULL, цільова ознака |

Технічний `row_id BIGINT IDENTITY` — surrogate PK; надійного природного ключа в джерелі немає. Для полів без бізнес-ліміту використовуємо TEXT. Грошовий баланс має NUMERIC(14,2), навіть якщо джерело містить лише цілі числа. Датасет не містить повної дати або timestamp події: **не створюємо фальшиві DATE/TIMESTAMPTZ**. `month` і `day` окремо не визначають дату без року.

## 2. Staging, COPY FROM STDIN, clean DDL

In [2]:
cols = list(df_source.columns)
assert cols == ['age','job','marital','education','default','balance','housing','loan','contact','day','month','duration','campaign','pdays','previous','poutcome','y']
raw_cols = [f'{c}_raw' for c in cols]
qcols = ', '.join(f'"{c}" TEXT' for c in raw_cols)
with engine.begin() as conn:
    conn.execute(text('DROP TABLE IF EXISTS hw3_bank_clean CASCADE'))
    conn.execute(text('DROP TABLE IF EXISTS hw3_bank_staging CASCADE'))
    conn.execute(text(f'CREATE TABLE hw3_bank_staging ({qcols})'))
raw = engine.raw_connection()
try:
    with raw.cursor() as cur, open(CSV_PATH, 'r', encoding='utf-8', newline='') as f:
        cur.copy_expert(f"COPY hw3_bank_staging ({', '.join(raw_cols)}) FROM STDIN WITH (FORMAT csv, HEADER true, DELIMITER ';')", f)
    raw.commit()
finally:
    raw.close()
print(pd.read_sql('SELECT COUNT(*) AS staging_rows FROM hw3_bank_staging', engine).to_string(index=False))

 staging_rows
        45211


In [3]:
ddl = """
CREATE TABLE hw3_bank_clean (
 row_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
 age SMALLINT NOT NULL CHECK (age BETWEEN 18 AND 100),
 job TEXT, marital TEXT, education TEXT,
 has_default BOOLEAN, balance NUMERIC(14,2), has_housing BOOLEAN, has_loan BOOLEAN,
 contact TEXT, contact_day SMALLINT NOT NULL CHECK (contact_day BETWEEN 1 AND 31),
 contact_month TEXT NOT NULL,
 duration INTEGER NOT NULL CHECK (duration >= 0),
 campaign INTEGER NOT NULL CHECK (campaign >= 1),
 pdays INTEGER CHECK (pdays IS NULL OR pdays >= 0),
 previous INTEGER NOT NULL CHECK (previous >= 0),
 poutcome TEXT,
 subscribed BOOLEAN NOT NULL
)
"""
with engine.begin() as conn:
    conn.execute(text(ddl))
    conn.execute(text("""
    INSERT INTO hw3_bank_clean (age,job,marital,education,has_default,balance,has_housing,has_loan,
      contact,contact_day,contact_month,duration,campaign,pdays,previous,poutcome,subscribed)
    SELECT
      TRIM(age_raw)::SMALLINT,
      NULLIF(NULLIF(TRIM(job_raw),''),'unknown'),
      NULLIF(NULLIF(TRIM(marital_raw),''),'unknown'),
      NULLIF(NULLIF(TRIM(education_raw),''),'unknown'),
      CASE TRIM(default_raw) WHEN 'yes' THEN TRUE WHEN 'no' THEN FALSE ELSE NULL END,
      NULLIF(TRIM(balance_raw),'')::NUMERIC(14,2),
      CASE TRIM(housing_raw) WHEN 'yes' THEN TRUE WHEN 'no' THEN FALSE ELSE NULL END,
      CASE TRIM(loan_raw) WHEN 'yes' THEN TRUE WHEN 'no' THEN FALSE ELSE NULL END,
      NULLIF(NULLIF(TRIM(contact_raw),''),'unknown'),
      TRIM(day_raw)::SMALLINT,
      TRIM(month_raw),
      TRIM(duration_raw)::INTEGER,
      TRIM(campaign_raw)::INTEGER,
      NULLIF(NULLIF(TRIM(pdays_raw),''),'-1')::INTEGER,
      TRIM(previous_raw)::INTEGER,
      NULLIF(NULLIF(TRIM(poutcome_raw),''),'unknown'),
      CASE TRIM(y_raw) WHEN 'yes' THEN TRUE ELSE FALSE END
    FROM hw3_bank_staging
    WHERE TRIM(age_raw) ~ '^[0-9]+$' AND TRIM(age_raw)::INTEGER BETWEEN 18 AND 100
      AND TRIM(day_raw) ~ '^[0-9]+$' AND TRIM(day_raw)::INTEGER BETWEEN 1 AND 31
      AND TRIM(duration_raw) ~ '^[0-9]+$' AND TRIM(campaign_raw) ~ '^[0-9]+$'
      AND TRIM(previous_raw) ~ '^[0-9]+$' AND TRIM(pdays_raw) ~ '^(-1|[0-9]+)$'
      AND TRIM(balance_raw) ~ '^-?[0-9]+$'
      AND TRIM(duration_raw)::INTEGER >= 0 AND TRIM(campaign_raw)::INTEGER >= 1
      AND TRIM(y_raw) IN ('yes','no') AND TRIM(month_raw) <> ''
    """))
print('Typed table created')

Typed table created


**Політика якості:** порожні та `unknown` категорії нормалізуються до SQL NULL; `pdays=-1` означає відсутність попереднього контакту, тому теж стає NULL. Не видаляємо від'ємні `balance`, оскільки це може бути справжній борг. Відкидаємо тільки рядки, які порушують перелічені у WHERE критичні доменні правила; кількість втрат перевіряємо нижче. `TRIM` застосовується до сирих текстових значень.

## 3. Data-quality audit

In [4]:
counts = pd.read_sql("SELECT 'staging' layer, COUNT(*) n_rows FROM hw3_bank_staging UNION ALL SELECT 'clean', COUNT(*) FROM hw3_bank_clean", engine)
display(counts)
a,b = map(int, counts.n_rows)
print(f'Висновок: staging={a}, clean={b}, відфільтровано={a-b}. Правило відбору — WHERE у INSERT: валідні age/day/duration/campaign/previous/pdays/balance/y/month; CHECK у фінальній таблиці.')

,layer,n_rows
0,staging,45211
1,clean,45211


Висновок: staging=45211, clean=45211, відфільтровано=0. Правило відбору — WHERE у INSERT: валідні age/day/duration/campaign/previous/pdays/balance/y/month; CHECK у фінальній таблиці.


In [5]:
dup = pd.read_sql("""
SELECT COUNT(*) total_rows,
 COUNT(DISTINCT (age,job,marital,education,has_default,balance,has_housing,has_loan,contact,contact_day,contact_month,duration,campaign,pdays,previous,poutcome,subscribed)) distinct_feature_tuples
FROM hw3_bank_clean""",engine)
dup['potential_duplicates'] = dup.total_rows - dup.distinct_feature_tuples
display(dup)
print('Висновок: виявлено',int(dup.potential_duplicates.iloc[0]),'точних збігів за всіма бізнес-полями. Без ID клієнта та дати контакту не можна довести, що це повтори однієї події; автоматично не видаляємо.')

,total_rows,distinct_feature_tuples,potential_duplicates
0,45211,45211,0


Висновок: виявлено 0 точних збігів за всіма бізнес-полями. Без ID клієнта та дати контакту не можна довести, що це повтори однієї події; автоматично не видаляємо.


In [6]:
nulls = pd.read_sql("""
SELECT COUNT(*) n,
 ROUND(100.0*COUNT(*) FILTER(WHERE job IS NULL)/NULLIF(COUNT(*),0),2) job_null_pct,
 ROUND(100.0*COUNT(*) FILTER(WHERE education IS NULL)/NULLIF(COUNT(*),0),2) education_null_pct,
 ROUND(100.0*COUNT(*) FILTER(WHERE contact IS NULL)/NULLIF(COUNT(*),0),2) contact_null_pct,
 ROUND(100.0*COUNT(*) FILTER(WHERE pdays IS NULL)/NULLIF(COUNT(*),0),2) pdays_null_pct,
 ROUND(100.0*COUNT(*) FILTER(WHERE poutcome IS NULL)/NULLIF(COUNT(*),0),2) poutcome_null_pct,
 ROUND(100.0*COUNT(*) FILTER(WHERE balance IS NULL)/NULLIF(COUNT(*),0),2) balance_null_pct
FROM hw3_bank_clean""",engine)
display(nulls)
print('Висновок: NULL% показує пропуски після нормалізації. pdays=NULL має спеціальний бізнес-зміст (не контактували), а не лише втрату даних.')
sentinels = pd.read_sql("""
SELECT COUNT(*) FILTER(WHERE LOWER(TRIM(job_raw)) IN ('unknown','?','')) job_marker,
 COUNT(*) FILTER(WHERE LOWER(TRIM(education_raw)) IN ('unknown','?','')) education_marker,
 COUNT(*) FILTER(WHERE LOWER(TRIM(contact_raw)) IN ('unknown','?','')) contact_marker,
 COUNT(*) FILTER(WHERE LOWER(TRIM(poutcome_raw)) IN ('unknown','?','')) poutcome_marker,
 COUNT(*) FILTER(WHERE TRIM(pdays_raw)='-1') pdays_minus_one,
 COUNT(*) FILTER(WHERE TRIM(balance_raw)='') empty_balance
FROM hw3_bank_staging""",engine)
display(sentinels)
print('Висновок: наведено фактичну кількість sentinel-маркерів у сирому CSV; нульове значення не гарантує відсутності інших проблем якості.')

,n,job_null_pct,education_null_pct,contact_null_pct,pdays_null_pct,poutcome_null_pct,balance_null_pct
0,45211,0.64,4.11,28.8,81.74,81.75,0.0


Висновок: NULL% показує пропуски після нормалізації. pdays=NULL має спеціальний бізнес-зміст (не контактували), а не лише втрату даних.


,job_marker,education_marker,contact_marker,poutcome_marker,pdays_minus_one,empty_balance
0,288,1857,13020,36959,36954,0


Висновок: наведено фактичну кількість sentinel-маркерів у сирому CSV; нульове значення не гарантує відсутності інших проблем якості.


## 4. Розподіли, статистика та outliers

In [7]:
dist = pd.read_sql("""SELECT job, COUNT(*) n_rows, ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER(),2) pct_rows FROM hw3_bank_clean GROUP BY job ORDER BY n_rows DESC, job LIMIT 10""",engine)
display(dist)
print(f"Висновок: найбільша категорія job={dist.iloc[0]['job']}, {dist.iloc[0]['n_rows']} записів ({dist.iloc[0]['pct_rows']}%). Рідкісні категорії потребують уваги під час кодування features.")
stats = pd.read_sql('SELECT COUNT(balance) n_non_null, MIN(balance) min_value, MAX(balance) max_value, ROUND(AVG(balance),2) avg_value, ROUND(STDDEV_SAMP(balance),2) sd_value FROM hw3_bank_clean',engine)
display(stats)
print(f"Висновок: balance лежить між {stats.min_value.iloc[0]} та {stats.max_value.iloc[0]}; від'ємні значення не вважаємо помилкою автоматично.")
outliers = pd.read_sql('SELECT row_id, age, job, balance, subscribed FROM hw3_bank_clean WHERE balance < -10000 OR balance > 100000 ORDER BY ABS(balance) DESC LIMIT 20',engine)
display(outliers)
print(f'Висновок: domain-check (balance < -10000 або > 100000) повернув {len(outliers)} рядків (максимум 20 показано). Це кандидати на перевірку, а не підстава для автоматичного видалення. Рекомендація: flag + перевірка джерела.')

,job,n_rows,pct_rows
0,blue-collar,9732,21.53
1,management,9458,20.92
2,technician,7597,16.80
3,admin.,5171,11.44
4,services,4154,9.19
5,retired,2264,5.01
6,self-employed,1579,3.49
7,entrepreneur,1487,3.29
8,unemployed,1303,2.88
9,housemaid,1240,2.74


Висновок: найбільша категорія job=blue-collar, 9732 записів (21.53%). Рідкісні категорії потребують уваги під час кодування features.


,n_non_null,min_value,max_value,avg_value,sd_value
0,45211,-8019.0,102127.0,1362.27,3044.77


Висновок: balance лежить між -8019.0 та 102127.0; від'ємні значення не вважаємо помилкою автоматично.


,row_id,age,job,balance,subscribed
0,39990,51,management,102127.0,False


Висновок: domain-check (balance < -10000 або > 100000) повернув 1 рядків (максимум 20 показано). Це кандидати на перевірку, а не підстава для автоматичного видалення. Рекомендація: flag + перевірка джерела.


## 5. Десять DQL / EDA-запитів

### Query 1

**Business-question:** Які професії найпоширеніші?

```sql
SELECT job, COUNT(*) n FROM hw3_bank_clean WHERE job IS NOT NULL GROUP BY job ORDER BY n DESC, job LIMIT 5;
```

**Interpretation:** Професія blue-collar має 9 732 записи, management — 9 458, technician — 7 597. Це найчисленніші групи контактів; частота контакту не означає вищу конверсію.

In [8]:
result_1 = pd.read_sql('SELECT job, COUNT(*) n FROM hw3_bank_clean WHERE job IS NOT NULL GROUP BY job ORDER BY n DESC, job LIMIT 5', engine)
display(result_1)
print("Interpretation: Професія blue-collar має 9 732 записи, management — 9 458, technician — 7 597. Це найчисленніші групи контактів; частота контакту не означає вищу конверсію.")

,job,n
0,blue-collar,9732
1,management,9458
2,technician,7597
3,admin.,5171
4,services,4154


Interpretation: Професія blue-collar має 9 732 записи, management — 9 458, technician — 7 597. Це найчисленніші групи контактів; частота контакту не означає вищу конверсію.


### Query 2

**Business-question:** Яка частка підписок для різних професій?

```sql
SELECT job, COUNT(*) n, ROUND(100.0*AVG(subscribed::INT),2) conversion_pct FROM hw3_bank_clean GROUP BY job ORDER BY conversion_pct DESC NULLS LAST LIMIT 10;
```

**Interpretation:** Найвищу частку підписок серед показаних категорій мають student (28,68%, 938 записів) і retired (22,79%, 2 264 записи). Порівнювати лише відсотки без розміру групи некоректно; це описова асоціація, не причинність.

In [9]:
result_2 = pd.read_sql('SELECT job, COUNT(*) n, ROUND(100.0*AVG(subscribed::INT),2) conversion_pct FROM hw3_bank_clean GROUP BY job ORDER BY conversion_pct DESC NULLS LAST LIMIT 10', engine)
display(result_2)
print("Interpretation: Найвищу частку підписок серед показаних категорій мають student (28,68%, 938 записів) і retired (22,79%, 2 264 записи). Порівнювати лише відсотки без розміру групи некоректно; це описова асоціація, не причинність.")

,job,n,conversion_pct
0,student,938,28.68
1,retired,2264,22.79
2,unemployed,1303,15.50
3,management,9458,13.76
4,admin.,5171,12.20
5,self-employed,1579,11.84
6,None,288,11.81
7,technician,7597,11.06
8,services,4154,8.88
9,housemaid,1240,8.79


Interpretation: Найвищу частку підписок серед показаних категорій мають student (28,68%, 938 записів) і retired (22,79%, 2 264 записи). Порівнювати лише відсотки без розміру групи некоректно; це описова асоціація, не причинність.


### Query 3

**Business-question:** Чи відрізняється частка підписок між клієнтами 25–45 років із особистим кредитом та без нього?

```sql
SELECT has_loan, COUNT(*) AS n, ROUND(100.0*AVG(subscribed::INT),2) AS conversion_pct FROM hw3_bank_clean WHERE age BETWEEN 25 AND 45 AND (NOT (has_loan IS NULL) OR has_loan = FALSE) GROUP BY has_loan ORDER BY has_loan;
```

**Interpretation:** Після виконання запиту порівняйте conversion_pct для has_loan = true та false і вкажіть різницю у відсоткових пунктах. Обидві групи обмежено віком 25–45 років; висновок стосується лише цієї підвибірки.

In [10]:
result_3 = pd.read_sql(text("""SELECT has_loan, COUNT(*) AS n, ROUND(100.0*AVG(subscribed::INT),2) AS conversion_pct FROM hw3_bank_clean WHERE age BETWEEN 25 AND 45 AND (NOT (has_loan IS NULL) OR has_loan = FALSE) GROUP BY has_loan ORDER BY has_loan"""), engine)
display(result_3)
print("Interpretation: Після виконання запиту порівняйте conversion_pct для has_loan = true та false і вкажіть різницю у відсоткових пунктах. Обидві групи обмежено віком 25–45 років; висновок стосується лише цієї підвибірки.")

,has_loan,n,conversion_pct
0,False,25143,11.75
1,True,4811,6.82


Interpretation: Після виконання запиту порівняйте conversion_pct для has_loan = true та false і вкажіть різницю у відсоткових пунктах. Обидві групи обмежено віком 25–45 років; висновок стосується лише цієї підвибірки.


### Query 4

**Business-question:** Які категорії job починаються з blue?

```sql
SELECT DISTINCT job FROM hw3_bank_clean WHERE job ILIKE 'blue%' ORDER BY job LIMIT 20;
```

**Interpretation:** Запит знаходить одну категорію — blue-collar. DISTINCT усуває повторення значень професії, а не дублікати записів клієнтів.

In [11]:
from sqlalchemy import text

result_4 = pd.read_sql(
    text("""
        SELECT DISTINCT job
        FROM hw3_bank_clean
        WHERE job ILIKE 'blue%'
        ORDER BY job
        LIMIT 20
    """),
    engine
)


display(result_4)
print("Interpretation: Запит знаходить одну категорію — blue-collar. DISTINCT усуває повторення значень професії, а не дублікати записів клієнтів.")

,job
0,blue-collar


Interpretation: Запит знаходить одну категорію — blue-collar. DISTINCT усуває повторення значень професії, а не дублікати записів клієнтів.


### Query 5

**Business-question:** Яка частка записів має невідомий канал контакту?

```sql
SELECT CASE WHEN contact IS NULL THEN 'missing' ELSE 'known' END AS contact_status, COUNT(*) AS n, ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (),2) AS pct_rows FROM hw3_bank_clean GROUP BY contact_status ORDER BY n DESC;
```

**Interpretation:** Із 45 211 контактів 13 020 (28,80%) мають невідомий канал, а 32 191 (71,20%) — відомий. Висока частка пропусків може впливати на порівняння ефективності каналів і потребує окремої категорії або перевірки джерела.

In [12]:
result_5 = pd.read_sql(text("""SELECT CASE WHEN contact IS NULL THEN 'missing' ELSE 'known' END AS contact_status, COUNT(*) AS n, ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (),2) AS pct_rows FROM hw3_bank_clean GROUP BY contact_status ORDER BY n DESC"""), engine)
display(result_5)
print("Interpretation: Із 45 211 контактів 13 020 (28,80%) мають невідомий канал, а 32 191 (71,20%) — відомий. Висока частка пропусків може впливати на порівняння ефективності каналів і потребує окремої категорії або перевірки джерела.")

,contact_status,n,pct_rows
0,known,32191,71.2
1,missing,13020,28.8


Interpretation: Із 45 211 контактів 13 020 (28,80%) мають невідомий канал, а 32 191 (71,20%) — відомий. Висока частка пропусків може впливати на порівняння ефективності каналів і потребує окремої категорії або перевірки джерела.


### Query 6

**Business-question:** Які значення кількості днів від попереднього контакту трапляються найчастіше (−1 — попереднього контакту немає)?

```sql
SELECT COALESCE(pdays,-1) AS days_since_previous, COUNT(*) AS n FROM hw3_bank_clean GROUP BY COALESCE(pdays,-1) ORDER BY n DESC LIMIT 10;
```

**Interpretation:** Значення −1, яке означає відсутність попереднього контакту, трапляється 36 954 рази (≈81,74% усіх записів). Серед записів із попереднім контактом найчастіше трапляється 182 дні (167 випадків); −1 не можна інтерпретувати як від’ємний інтервал.

In [13]:
result_6 = pd.read_sql(text("""SELECT COALESCE(pdays,-1) AS days_since_previous, COUNT(*) AS n FROM hw3_bank_clean GROUP BY COALESCE(pdays,-1) ORDER BY n DESC LIMIT 10"""), engine)
display(result_6)
print("Interpretation: Значення −1, яке означає відсутність попереднього контакту, трапляється 36 954 рази (≈81,74% усіх записів). Серед записів із попереднім контактом найчастіше трапляється 182 дні (167 випадків); −1 не можна інтерпретувати як від’ємний інтервал.")

,days_since_previous,n
0,-1,36954
1,182,167
2,92,147
3,183,126
4,91,126
5,181,117
6,370,99
7,184,85
8,364,77
9,95,74


Interpretation: Значення −1, яке означає відсутність попереднього контакту, трапляється 36 954 рази (≈81,74% усіх записів). Серед записів із попереднім контактом найчастіше трапляється 182 дні (167 випадків); −1 не можна інтерпретувати як від’ємний інтервал.


### Query 7

**Business-question:** Які рядки мають відмінність між прапорцями житлового та особистого кредиту?

```sql
SELECT row_id, has_housing, has_loan FROM hw3_bank_clean WHERE has_housing IS DISTINCT FROM has_loan ORDER BY row_id LIMIT 10;
```

**Interpretation:** Перші десять записів із різними значеннями прапорців мають has_housing=true і has_loan=false. LIMIT 10 показує приклади, але не дає загальної кількості таких контактів; різні кредитні продукти не є помилкою даних.

In [14]:
result_7 = pd.read_sql('SELECT row_id, has_housing, has_loan FROM hw3_bank_clean WHERE has_housing IS DISTINCT FROM has_loan ORDER BY row_id LIMIT 10', engine)
display(result_7)
print("Interpretation: Перші десять записів із різними значеннями прапорців мають has_housing=true і has_loan=false. LIMIT 10 показує приклади, але не дає загальної кількості таких контактів; різні кредитні продукти не є помилкою даних.")

,row_id,has_housing,has_loan
0,1,True,False
1,2,True,False
2,4,True,False
3,6,True,False
4,8,True,False
5,9,True,False
6,10,True,False
7,11,True,False
8,12,True,False
9,13,True,False


Interpretation: Перші десять записів із різними значеннями прапорців мають has_housing=true і has_loan=false. LIMIT 10 показує приклади, але не дає загальної кількості таких контактів; різні кредитні продукти не є помилкою даних.


### Query 8

**Business-question:** Які місяці мають найбільше контактів?

```sql
SELECT contact_month, COUNT(*) n FROM hw3_bank_clean GROUP BY contact_month ORDER BY n DESC LIMIT 5;
```

**Interpretation:** Найбільше контактів зафіксовано в травні (13 766), липні (6 895) та серпні (6 247). Це розподіл кількості контактів, а не доказ сезонного впливу на ймовірність підписки.

In [15]:
result_8 = pd.read_sql('SELECT contact_month, COUNT(*) n FROM hw3_bank_clean GROUP BY contact_month ORDER BY n DESC LIMIT 5', engine)
display(result_8)
print("Interpretation: Найбільше контактів зафіксовано в травні (13 766), липні (6 895) та серпні (6 247). Це розподіл кількості контактів, а не доказ сезонного впливу на ймовірність підписки.")

,contact_month,n
0,may,13766
1,jul,6895
2,aug,6247
3,jun,5341
4,nov,3970


Interpretation: Найбільше контактів зафіксовано в травні (13 766), липні (6 895) та серпні (6 247). Це розподіл кількості контактів, а не доказ сезонного впливу на ймовірність підписки.


### Query 9

**Business-question:** Чи змінюється частка підписок із кількістю спроб контакту?

```sql
SELECT CASE WHEN campaign IN (1,2) THEN '1-2' WHEN campaign BETWEEN 3 AND 5 THEN '3-5' ELSE '6+' END attempts, COUNT(*) n, ROUND(100.0*AVG(subscribed::INT),2) conversion_pct FROM hw3_bank_clean GROUP BY attempts ORDER BY attempts;
```

**Interpretation:** Для 1–2 спроб частка підписок становить 13,19% (30 049 записів), для 3–5 — 9,94% (10 807), для 6+ — 5,81% (4 355). Зменшення частки при більшій кількості спроб є асоціацією, а не доказом негативного ефекту повторних дзвінків.

In [16]:
result_9 = pd.read_sql("SELECT CASE WHEN campaign IN (1,2) THEN '1-2' WHEN campaign BETWEEN 3 AND 5 THEN '3-5' ELSE '6+' END attempts, COUNT(*) n, ROUND(100.0*AVG(subscribed::INT),2) conversion_pct FROM hw3_bank_clean GROUP BY attempts ORDER BY attempts", engine)
display(result_9)
print("Interpretation: Для 1–2 спроб частка підписок становить 13,19% (30 049 записів), для 3–5 — 9,94% (10 807), для 6+ — 5,81% (4 355). Зменшення частки при більшій кількості спроб є асоціацією, а не доказом негативного ефекту повторних дзвінків.")

,attempts,n,conversion_pct
0,1-2,30049,13.19
1,3-5,10807,9.94
2,6+,4355,5.81


Interpretation: Для 1–2 спроб частка підписок становить 13,19% (30 049 записів), для 3–5 — 9,94% (10 807), для 6+ — 5,81% (4 355). Зменшення частки при більшій кількості спроб є асоціацією, а не доказом негативного ефекту повторних дзвінків.


### Query 10

**Business-question:** Як пов’язаний результат попередньої кампанії з підпискою?

```sql
SELECT COALESCE(poutcome,'unknown') previous_outcome, COUNT(*) n, ROUND(100.0*AVG(subscribed::INT),2) conversion_pct FROM hw3_bank_clean WHERE subscribed IS NOT NULL GROUP BY COALESCE(poutcome,'unknown') ORDER BY conversion_pct DESC;
```

**Interpretation:** Після попереднього успіху конверсія становить 64,73% (1 511 записів), після невдачі — 12,61% (4 901). Для unknown вона становить 9,16% (36 959), тому історія попередніх кампаній потенційно корисна для прогнозу, якщо доступна до нового контакту.

In [17]:
result_10 = pd.read_sql("SELECT COALESCE(poutcome,'unknown') previous_outcome, COUNT(*) n, ROUND(100.0*AVG(subscribed::INT),2) conversion_pct FROM hw3_bank_clean WHERE subscribed IS NOT NULL GROUP BY COALESCE(poutcome,'unknown') ORDER BY conversion_pct DESC", engine)
display(result_10)
print("Interpretation: Після попереднього успіху конверсія становить 64,73% (1 511 записів), після невдачі — 12,61% (4 901). Для unknown вона становить 9,16% (36 959), тому історія попередніх кампаній потенційно корисна для прогнозу, якщо доступна до нового контакту.")

,previous_outcome,n,conversion_pct
0,success,1511,64.73
1,other,1840,16.68
2,failure,4901,12.61
3,unknown,36959,9.16


Interpretation: Після попереднього успіху конверсія становить 64,73% (1 511 записів), після невдачі — 12,61% (4 901). Для unknown вона становить 9,16% (36 959), тому історія попередніх кампаній потенційно корисна для прогнозу, якщо доступна до нового контакту.


### Перевірка покриття SQL

- Explicit SELECT columns: Q1–Q10; `WHERE` AND/OR/NOT із дужками: Q3; `BETWEEN`: Q3, Q9; `IN`: Q9; `ILIKE`: Q4; `IS NULL`: Q5; `IS NOT NULL`: Q1, Q10; `COALESCE`: Q6, Q10; `IS DISTINCT FROM`: Q7; `DISTINCT`: Q4; `ORDER BY` + `LIMIT`: Q1–Q4, Q6–Q8; `GROUP BY`: Q1–Q3, Q5–Q6, Q8–Q10.

## 6. Reflection (≈220 слів)

Цей набір даних показує, чому підготовка даних для машинного навчання починається з перевірки їхнього значення, а не лише формату. У вихідному файлі деякі категоріальні поля використовують `unknown` замість справжнього пропуску. Я нормалізувала такі маркери до NULL, щоб не змішувати невідомі значення з валідними категоріями. Окремо розглянула `pdays = -1`: це означає, що попереднього контакту не було, тому таке значення не слід трактувати як звичайну кількість днів. Дублікати перевіряла за всіма бізнес-атрибутами, проте без ідентифікатора клієнта неможливо довести, що однакові записи є помилковими повтореннями.

Для задачі **бінарної класифікації** цільовою змінною буде `subscribed`: чи оформить клієнт строковий депозит. Потенційні ознаки — вік, професія, освіта, сімейний стан, баланс, наявність кредитів та історія попередніх контактів. Категоріальні ознаки потрібно закодувати, а пропуски обробити з урахуванням їхньої семантики. Поле `duration` не можна використовувати, якщо прогноз робиться до дзвінка: фактична тривалість стає відомою лише після його завершення й спричиняє **data leakage**. Також слід перевірити доступність інших ознак на момент прогнозування.

Data engineer я запропонувала б додати унікальні ідентифікатори клієнта та контакту, повний timestamp, опис допустимих категорій, правила для sentinel-значень і автоматичні перевірки типів, діапазонів та дублікатів. Важливо зберігати сирий шар незмінним, а перетворення версіонувати. Для чесної оцінки моделі бажано використовувати часовий поділ даних, якщо з’являться достовірні дати подій.

## Завершальна перевірка

Після виконання всіх клітинок перевірте: версію PostgreSQL, 45 211 рядків у staging, наявність clean, усі 10 результатів, а також відсутність помилок. У Colab скачайте `data/bank-full.csv` за потреби; CSV відтворюється автоматично з офіційного джерела. Збережіть notebook з output клітинок перед здачею.

In [18]:
with engine.connect() as conn:
    staging_n = conn.execute(text('SELECT COUNT(*) FROM hw3_bank_staging')).scalar_one()
    clean_n = conn.execute(text('SELECT COUNT(*) FROM hw3_bank_clean')).scalar_one()
assert staging_n == 45211
assert clean_n > 0
assert all(f'result_{i}' in globals() for i in range(1,11))
print('FINAL CHECK PASSED:', {'staging':staging_n,'clean':clean_n,'dql_queries':10})

FINAL CHECK PASSED: {'staging': 45211, 'clean': 45211, 'dql_queries': 10}
